# Introduction: PyPSA & linopy

<img src="https://docs.pypsa.org/latest/assets/logo/logo-primary-light.svg#only-light" width="260px" />

`PyPSA` is an open source Python package for simulating and optimising modern energy systems that include features such as

- conventional generators with unit commitment (ramp-up, ramp-down, start-up, shut-down),
- time-varying wind and solar generation,
- energy storage with efficiency losses and inflow/spillage for hydroelectricity
- coupling to other energy sectors (electricity, transport, heat, industry),
- conversion between energy carriers (e.g. electricity to hydrogen),
- transmission networks (AC, DC, other fuels)

PyPSA can be used for a variety of problem types (e.g. electricity market modelling, long-term investment planning, transmission network expansion planning), and is designed to scale well with large networks and long time series.

Compared to building power system by hand in `linopy`, PyPSA does the following things for you:

- manage data inputs
- build optimisation problem
- communicate with the solver
- retrieve and process optimisation results
- manage data outputs

### Dependencies

- `pandas` for storing data about network components and time series
- `numpy` and `scipy` for linear algebra and sparse matrix calculations
- `matplotlib` and `cartopy` for plotting on a map
- `networkx` for network calculations
- `linopy` for handling optimisation problems

:::{note}
Documentation for this package is available at https://docs.pypsa.org/
:::

:::{note}
**On the workshop JupyterHub, everything is already installed** - just log in and run the cells below, nothing to set up.

No hub access? You can also run this notebook in your browser on [Google Colab](https://colab.research.google.com/github/pypsalabs/workshop-202610/blob/main/berlin/01-pypsa-intro.ipynb) (a Google account is required). On Colab, first run this in a cell at the top:

```sh
!pip install pypsa
```
:::

## Basic components

| Component | Description |
| --- | --- |
| [Network](https://docs.pypsa.org/latest/user-guide/design/#network-components) | Container for all components. |
| [Bus](https://docs.pypsa.org/latest/user-guide/components/buses/) | Node where components attach. |
| [Carrier](https://docs.pypsa.org/latest/user-guide/components/carriers/) | Energy carrier or technology (e.g. electricity, hydrogen, gas, coal, oil, biomass, on-/offshore wind, solar). Can track properties such as specific carbon dioxide emissions or nice names and colors for plots. |
| [Load](https://docs.pypsa.org/latest/user-guide/components/loads/) | Energy consumer (e.g. electricity demand). |
| [Generator](https://docs.pypsa.org/latest/user-guide/components/generators/) | Generator (e.g. power plant, wind turbine, PV panel). |
| [Line](https://docs.pypsa.org/latest/user-guide/components/lines/) | Power distribution and transmission lines (overhead and cables). |
| [Link](https://docs.pypsa.org/latest/user-guide/components/links/) | Links connect two buses with controllable energy flow, direction-control and losses. They can be used to model: <ul><li>HVDC links</li><li>HVAC lines (neglecting KVL, only net transfer capacities (NTCs))</li><li>conversion between carriers (e.g. electricity to hydrogen in electrolysis)</li></ul> |
| [StorageUnit](https://docs.pypsa.org/latest/user-guide/components/storage-units/) | Storage with fixed nominal energy-to-power ratio. |
| [Store](https://docs.pypsa.org/latest/user-guide/components/stores/) | Storage with separately extendable energy capacity. |
| [GlobalConstraint](https://docs.pypsa.org/latest/user-guide/components/global-constraints/) | Constraints affecting many components at once, such as emission limits. |

:::{note}
Links in the table lead to documentation for each component.
:::

## Getting started

### Simple electricity market problem

generator 1: "gas" -- marginal cost 70 EUR/MWh -- capacity 50 MW

generator 2: "nuclear" -- marginal cost 10 EUR/MWh -- capacity 100 MW

load: "Small town" -- demand 120 MW

single time step ("now")

single node ("Springfield")

### Building a basic network

In [ ]:
# By convention, PyPSA is imported without an alias:
import pypsa

In [ ]:
# First, we create a network object which serves as the container for all components
n = pypsa.Network()

In [ ]:
n

The second component we need are buses. **Buses** are the fundamental nodes of the network, to which all other components like loads, generators and transmission lines attach. They enforce energy conservation for all elements feeding in and out of it (i.e. Kirchhoff’s Current Law).

<img src="https://docs.pypsa.org/latest/assets/images/buses.png" width="500px" />

Components can be added to the network `n` using the `n.add()` function. It takes the component name as a first argument, the name of the component as a second argument and possibly further parameters as keyword arguments. Let's use this function to add a single bus, "Springfield", to our network:

In [ ]:
n.add("Bus", "Springfield", v_nom=380, carrier="AC")

For each class of components, the data describing the components is stored in a `pandas.DataFrame`. For example, all static data for buses is stored in `n.buses`

In [ ]:
n.buses

You see there are many more attributes than we specified while adding the buses; many of them are filled with default parameters which were added. You can look up the field description, defaults and status (required input, optional input, output) for buses here https://docs.pypsa.org/latest/user-guide/components/buses/, and analogous for all other components.

The `n.add()` function lets you add any component to the network object `n`:

In [ ]:
n.add(
    "Generator",
    "gas",
    bus="Springfield",
    p_nom_extendable=False,
    marginal_cost=70,  # €/MWh
    p_nom=50,  # MW
)
n.add(
    "Generator",
    "nuclear",
    bus="Springfield",
    p_nom_extendable=False,
    marginal_cost=10,  # €/MWh
    p_nom=100,  # MW
)

As a result, the `n.generators` DataFrame looks like this:

In [ ]:
n.generators

Next, we're going to add the electricity demand.

A positive value for `p_set` means consumption of power from the bus.

In [ ]:
n.add(
    "Load",
    "Small town",
    bus="Springfield",
    p_set=120,  # MW
)

In [ ]:
n.loads

## Optimisation

The design principle of PyPSA is that basically each component is associated with a set of variables and constraints that will be added to the optimisation model based on the input data stored for the components. 

For this dispatch problem, PyPSA will solve an optimisation problem that looks like this

\begin{equation}
\min_{g_{s,t};} \sum_{t,s} o_{s} g_{s,t}
\end{equation}
such that
\begin{align}
0 & \leq g_{s,t} \leq G_{s}  & \text{generation limits : generator} \\
D_t &= \sum_s g_{s,t} & \text{market clearing : bus} \\
\end{align}

**Decision variables:**

- $g_{s,t}$ is the generator dispatch of technology $s$ at time $t$

**Parameters:**

- $o_{s}$ is the marginal generation cost of technology $s$
- $G_{s}$ is the nominal capacity of technology $s$
- $D_t$ is the power demand in Springfield at time $t$


With all input data transferred into the PyPSA's data structure (`network`), we can now build and run the resulting optimisation problem. In PyPSA, building, solving and retrieving results from the optimisation model is contained in a single function call `n.optimize()`. This function optimizes dispatch and investment decisions for least cost adhering to the constraints defined in the network.

The `n.optimize()` function can take a variety of arguments. The most relevant for the moment is the choice of the solver (e.g. "highs" and "gurobi"). They need to be installed on your computer, to use them here!

In [ ]:
n.optimize(solver_name="highs")

Let's have a look at the results. The network object `n` contains now the objective value and the results for the decision variables.

In [ ]:
n.objective

Since the power flow and dispatch are generally time-varying quantities, these are stored in a different locations than e.g. `n.generators`. They are stored in `n.generators_t`. Thus, to find out the dispatch of the generators, run

In [ ]:
n.generators_t.p

In [ ]:
n.buses_t.marginal_price

In [ ]:
n.generators_t.mu_upper

### Explore the PyPSA model

Under the hood, a call to `n.optimize()` builds a `linopy` optimisation model, solves it with the specified solver, and then retrieves the results back into the PyPSA data structure. We can access this intermediate `linopy` model via `n.model`. This allows us to explore the optimisation model in more detail, for instance to see all variables and constraints that were added to the optimisation problem.

In [ ]:
n.model

In [ ]:
n.model.constraints

In [ ]:
n.model.constraints["Generator-fix-p-upper"]

In [ ]:
n.model.constraints["Bus-nodal_balance"]

In [ ]:
n.model.objective

We can also retrieve the shadow prices (duals) of individual constraints, e.g. of the upper generation limits:

In [ ]:
n.model.constraints["Generator-fix-p-upper"].dual.to_dataframe()

### Let's write optimization problem manually and reproduce the PyPSA model

PyPSA optimisation module is based on [Linopy](https://linopy.readthedocs.io/en/latest/index.html) --- an open-source framework for formulating, solving, and analyzing optimization problems with Python.

With Linopy, you can create optimization models within Python that consist of decision variables, constraints, and optimization objectives. You can then solve these instances using a variety of commercial and open-source solvers (specialised software).

[Linopy](https://github.com/pypsa/linopy) supports a wide range of problem types, including:

- **Linear programming**
- Integer programming
- Mixed-integer programming
- Quadratic programming


:::{note}
Documentation for this package is available at https://linopy.readthedocs.io.
:::

In [ ]:
# remove all constraints
for name in list(n.model.constraints):
    n.model.remove_constraints(name)

In [ ]:
n.model.constraints

In [ ]:
n.model.add_constraints(
    n.model["Generator-p"].sum(dim="name") == n.loads.p_set["Small town"],
    name="nodal_balance",
)

In [ ]:
n.model.add_constraints(n.model["Generator-p"].loc[:, "gas"] >= 0, name="p_lower_gas")

In [ ]:
n.model.add_constraints(
    n.model["Generator-p"].loc[:, "nuclear"] >= 0, name="p_lower_nuclear"
)

In [ ]:
n.model.add_constraints(
    n.model["Generator-p"].loc[:, "gas"] <= n.generators.p_nom.loc["gas"],
    name="p_upper_gas",
)

In [ ]:
n.model.add_constraints(
    n.model["Generator-p"].loc[:, "nuclear"] <= n.generators.p_nom.loc["nuclear"],
    name="p_upper_nuclear",
)

In [ ]:
# check that we did a good job
n.model.constraints

Let's ensure that we get the same results as when using the `n.optimize()` function

In [ ]:
n.optimize.solve_model(solver_name="highs")

In [ ]:
n.generators_t.p

In [ ]:
n.model

### Making use of Expressions

Besides variables and constraints, a `linopy` model has a third group: **Expressions**. These are *named* linear expressions, i.e. combinations of variables that are not (yet) turned into a constraint or an objective.

Registering an expression with `n.model.add_expressions(..., name=...)` makes it part of the model:

- it appears in the model summary,
- it can be reused when building further expressions, constraints or the objective,
- after solving, it evaluates to numbers via `.solution`.

This is handy for derived quantities you want to report, without recomputing them by hand from the decision variables.

In [ ]:
# total dispatch per snapshot, dims: (snapshot)
total_generation = n.model.add_expressions(
    n.model["Generator-p"].sum(dim="name"), name="total_generation"
)
total_generation

In [ ]:
# operational cost; a scalar expression that should coincide with the objective
operational_cost = n.model.add_expressions(
    (n.generators.marginal_cost * n.model["Generator-p"]).sum(), name="operational_cost"
)
operational_cost

In [ ]:
# the Expressions group is no longer empty. Expressions have no numerical value until the model is solved.
n.model

In [ ]:
n.model.solve(solver_name="highs")

In [ ]:
# all registered expressions evaluated at the optimum, as one xarray.Dataset
n.model.expressions.solution.to_pandas()

In [ ]:
# or one at a time; the cost expression reproduces the objective value
n.model.expressions["operational_cost"].solution.item() == n.model.objective.value

## Component objects in PyPSA v1

Since PyPSA v1.0, every component type also has a dedicated [`Components` object](https://docs.pypsa.org/latest/user-guide/components/), available through `n.components` or its short alias `n.c`. The `Components` class adds functionality beyond raw DataFrames, including shared methods and properties on [`pypsa.Components`](https://docs.pypsa.org/latest/api/components/components/) and features specific to individual component types.

The underlying pandas-based data structure remains unchanged. Each component object provides access to two stores:

- `.static`: one row per asset for values that do not vary across snapshots, such as nominal capacity or marginal cost;
- `.dynamic`: a dictionary-like collection of time-indexed DataFrames, such as dispatch or marginal prices.

These are exactly the same data exposed by the familiar plural and `_t` accessors. For example, `n.components.generators.static` is `n.generators`, while `n.components.generators.dynamic` is `n.generators_t`. Both styles are supported in PyPSA v1.

In [ ]:
# Access the Generators component object, which bundles component data and functionality
generators = n.components.generators
generators

In [ ]:
# Access time-dependent dispatch through the familiar `_t` accessor
n.generators_t.p

In [ ]:
# Access the same dispatch DataFrame through the Components API
generators.dynamic.p

In [ ]:
# Confirm that both accessors point to the exact same DataFrame, not a copy
n.generators_t.p is n.components.generators.dynamic.p

In [ ]:
# Select static generator attributes, with one row per asset
generators.static[["bus", "p_nom", "marginal_cost"]]

In [ ]:
# Inspect which generators are extendable and which have fixed capacities
print(generators.extendables)
print(generators.fixed)